# Setup and IResNet-100 smoke test

This stage validates the persisted environment and prepared data before candidate training. It intentionally stops on any contract violation.


In [ ]:
# Configuration
from pathlib import Path
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "face_name_metric_learning").is_dir(): REPO_ROOT = REPO_ROOT.parent
MANIFEST_PATH = REPO_ROOT / "data/processed/manifest.csv"
HDF5_PATH = REPO_ROOT / "data/processed/faces.h5"
ENVIRONMENT_RECORD_PATH = REPO_ROOT / "artifacts/face_name_metric_learning/setup_environment.json"
PRETRAINED_WEIGHTS_PATH = REPO_ROOT / "models/pretrained/iresnet100.pth"
PRETRAINED_MODULE = "pytorch_insightface.backbones.iresnet"
PRETRAINED_WEIGHT_IDENTIFIER = "pytorch-insightface:iresnet100:pretrained"
PRETRAINED_WEIGHT_SHA256 = None
EXPECTED_SPLIT_COUNTS = {"train": 45078, "validation": 9660, "test": 9660}
EXPECTED_CLASS_COUNT = 100
EMBEDDING_DIMENSION = 512
PINNED_TRAINING_PACKAGES = ("torch==2.7.1", "pytorch-metric-learning==2.8.1", "pytorch-insightface==0.7.3")


## Install and import the pinned runtime
Before continuing, install any reported requirements with the displayed `%pip` command.


In [ ]:
import sys
from importlib import metadata
missing_or_wrong = []
for requirement in PINNED_TRAINING_PACKAGES:
    name, expected = requirement.split("==", 1)
    try: actual = metadata.version(name)
    except metadata.PackageNotFoundError: actual = None
    if actual != expected: missing_or_wrong.append(requirement)
if missing_or_wrong:
    command = "install " + " ".join(missing_or_wrong)
    get_ipython().run_line_magic("pip", command)
    raise RuntimeError("Restart the kernel after installing the pinned runtime")


In [ ]:
import h5py
import torch
from face_name_metric_learning import collect_environment_record, load_pretrained_iresnet100, sha256_file, validate_manifest_hdf5, write_environment_record
assert REPO_ROOT.is_dir()
assert MANIFEST_PATH.is_file(), f"Missing manifest: {MANIFEST_PATH}"
assert HDF5_PATH.is_file(), f"Missing prepared faces: {HDF5_PATH}"
data_record = validate_manifest_hdf5(MANIFEST_PATH, HDF5_PATH, expected_split_counts=EXPECTED_SPLIT_COUNTS, expected_class_count=EXPECTED_CLASS_COUNT)
print(data_record)


## Load one pretrained image encoder and smoke-test one known HDF5 row
No split is augmented. A 512-dimensional row is normalized in float32 before the assertion.


In [ ]:
assert PRETRAINED_WEIGHTS_PATH.is_file(), f"Missing pretrained weights: {PRETRAINED_WEIGHTS_PATH}"
weight_sha256 = sha256_file(PRETRAINED_WEIGHTS_PATH)
if PRETRAINED_WEIGHT_SHA256 is not None: assert weight_sha256 == PRETRAINED_WEIGHT_SHA256
environment_record = collect_environment_record()
model = load_pretrained_iresnet100(PRETRAINED_WEIGHTS_PATH, module_name=PRETRAINED_MODULE, device=environment_record["device"], expected_sha256=PRETRAINED_WEIGHT_SHA256)
model.eval()
with h5py.File(HDF5_PATH, "r") as h5:
    known_index = int(h5["split_indices/validation"][0])
    image = torch.from_numpy(h5["images"][known_index:known_index + 1]).to(environment_record["device"])
with torch.inference_mode():
    embedding = model(image).detach().to(dtype=torch.float32, device="cpu").squeeze(0)
embedding = torch.nn.functional.normalize(embedding, p=2, dim=0)
assert embedding.shape == (EMBEDDING_DIMENSION,), embedding.shape
assert torch.isfinite(embedding).all()
assert torch.isclose(torch.linalg.vector_norm(embedding), torch.tensor(1.0), atol=1e-5)


In [ ]:
environment_record.update({"data": data_record, "paths": {"manifest": str(MANIFEST_PATH), "faces_hdf5": str(HDF5_PATH)}, "pretrained_weights": {"identifier": PRETRAINED_WEIGHT_IDENTIFIER, "sha256": weight_sha256}, "smoke_test": {"h5_index": known_index, "embedding_shape": list(embedding.shape), "embedding_dtype": str(embedding.dtype), "normalized": True}})
write_environment_record(ENVIRONMENT_RECORD_PATH, environment_record)
print(ENVIRONMENT_RECORD_PATH)
